# PyTorch — Cheat sheet

Referencia rápida para redes neuronales con **PyTorch**: tensores, autograd, módulos, pérdidas, optimizadores y bucle de entrenamiento.

Complementa los notebooks `01`–`03` de esta carpeta y las conversiones con pandas/NumPy en `04-pandas/04.06-numpy-pandas-pytorch-interop.ipynb`.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

## 1. Tensores

| Crear | Ejemplo |
|-------|---------|
| Desde lista | `torch.tensor([1.0, 2.0])` |
| Desde NumPy (copia) | `torch.tensor(arr)` |
| Desde NumPy (comparte memoria CPU) | `torch.from_numpy(arr)` |
| Ceros / unos | `torch.zeros(3, 4)`, `torch.ones(2, 2)` |
| Aleatorio | `torch.randn(5, 3)` |

Atributos útiles: `.shape`, `.dtype`, `.device`, `.T`, `.reshape(n, m)`, `.squeeze()`, `.unsqueeze(dim)`.

In [ ]:
x = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
x.shape          # torch.Size([2, 2])
x.to(device)     # mismo tensor en GPU si hay CUDA
x.float()        # cambiar dtype

## 2. Autograd (gradientes)

- `requires_grad=True` en tensores que participan en el entrenamiento.
- Tras `loss.backward()`, los gradientes están en `.grad` de cada parámetro.
- `optimizer.zero_grad()` antes de cada paso para no acumular gradientes viejos.

In [ ]:
w = torch.tensor(2.0, requires_grad=True)
loss = (w - 1) ** 2
loss.backward()
w.grad           # tensor(2.) — derivada de (w-1)² en w=2

with torch.no_grad():   # inferencia sin grafo
    y = w * 3

## 3. Capas y modelos (`nn.Module`)

| Capa | Uso típico |
|------|------------|
| `nn.Linear(in, out)` | Fully connected |
| `nn.ReLU()`, `nn.Sigmoid()` | Activaciones |
| `nn.Sequential(...)` | Apilar capas |

Salida de regresión: sin activación en la última capa.  
Clasificación: **logits** en la salida; la pérdida aplica sigmoid/softmax internamente.

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_in, n_hidden, n_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, n_hidden),
            nn.ReLU(),
            nn.Linear(n_hidden, n_out),
        )

    def forward(self, x):
        return self.net(x)

model = MLP(10, 32, 1).to(device)
list(model.parameters())   # pesos entrenables

## 4. Funciones de pérdida

| Tarea | Pérdida | Salida del modelo |
|-------|---------|-------------------|
| Regresión | `nn.MSELoss()` | Valor continuo |
| Binaria | `nn.BCEWithLogitsLoss()` | 1 logit por muestra |
| Multiclase | `nn.CrossEntropyLoss()` | `n_classes` logits (sin softmax manual) |

En inferencia binaria: `torch.sigmoid(logits)`. Multiclase: `logits.argmax(dim=1)`.

## 5. Optimizadores

```python
optimizer = optim.Adam(model.parameters(), lr=1e-3)
# Alternativas: optim.SGD(..., momentum=0.9), optim.RMSprop(...)
```

Un paso de entrenamiento: `zero_grad` → `forward` → `loss` → `backward` → `step`.

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# X, y: tensores en device con formas (batch, features) y (batch, 1) o (batch,)
X = torch.randn(32, 10, device=device)
y = torch.randn(32, 1, device=device)

model.train()
optimizer.zero_grad()
pred = model(X)
loss = criterion(pred, y)
loss.backward()
optimizer.step()
loss.item()   # escalar Python para imprimir o graficar

## 6. Bucle de entrenamiento (plantilla)

```python
for epoch in range(n_epochs):
    model.train()
    for X_batch, y_batch in dataloader:   # o tensores completos en ejemplos pequeños
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val), y_val)
```

## 7. Inferencia y guardado

| Acción | Código |
|--------|--------|
| Modo eval | `model.eval()` |
| Sin gradientes | `with torch.no_grad(): ...` |
| Guardar pesos | `torch.save(model.state_dict(), "model.pt")` |
| Cargar pesos | `model.load_state_dict(torch.load("model.pt", map_location=device))` |

Tensor → NumPy: `t.detach().cpu().numpy()` (ver notebook de interop en `04-pandas`).

## 8. Enlaces

- [Documentación PyTorch](https://pytorch.org/docs/stable/index.html)
- [Tutorial oficial](https://pytorch.org/tutorials/)
- Notebooks de esta carpeta: `01` regresión, `02` binaria, `03` multiclase